# Quantum Field Theory

This notebook covers 7 fundamental equations of Quantum Field Theory (QFT), from relativistic wave equations to non-linear classical field theories that model solitons and gauge dynamics.

Each section presents:
1. The defining equation with physical context
2. A symbolic solution
3. A numerical solution with visualization

All equations are registered in the `diff_eq_solver` equation registry.

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from diff_eq_solver import registry
from diff_eq_solver.visualizer import plot_ode_solution, plot_phase_portrait, plot_pde_heatmap, plot_pde_snapshots, plot_3d_surface, plot_comparison, plot_orbit, plot_special_function
from IPython.display import Math, display

## 1. Klein-Gordon Equation

The Klein-Gordon equation describes spin-0 particles in relativistic quantum mechanics:

$$\frac{\partial^2 \phi}{\partial t^2} - c^2 \nabla^2 \phi + \left(\frac{mc^2}{\hbar}\right)^2 \phi = 0$$

In natural units ($c = \hbar = 1$) with one spatial dimension:

$$\frac{\partial^2 \phi}{\partial t^2} - \frac{\partial^2 \phi}{\partial x^2} + m^2 \phi = 0$$

**Applications:** Scalar mesons (pions), Higgs boson field, inflationary cosmology (inflaton field), phonons in solid state physics.

In [ ]:
# Symbolic solution
eq = registry.get('klein_gordon_equation')
print("Klein-Gordon Equation:")
print(f"  Form: {eq.description}")
sol = eq.symbolic_solution()
print(f"  General solution: {sol}")

In [ ]:
# Numerical solution: wave packet dispersion (massive vs massless)
Nx = 300
Nt = 200
x = np.linspace(-15, 15, Nx)
t = np.linspace(0, 8, Nt)

# Gaussian wave packet: phi(x,0) = exp(-x^2/sigma^2) * cos(k0*x)
sigma = 1.5
k0 = 3.0
phi0 = np.exp(-x**2 / sigma**2) * np.cos(k0 * x)
dphi0 = np.zeros(Nx)  # initially at rest

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for idx, (mass, title) in enumerate([(0.0, 'Massless ($m=0$)'), (1.0, 'Massive ($m=1$)')]):
    result = eq.numerical_solve(
        t_span=(0, 8),
        u0=np.column_stack([phi0, dphi0]),
        x=x,
        t_eval=t,
        params={'m': mass, 'c': 1.0}
    )
    plot_pde_heatmap(result, ax=axes[idx],
        x_label='$x$', t_label='$t$',
        title=f'Klein-Gordon: {title}')

plt.suptitle('Wave Packet Dispersion: Massless vs Massive', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Dirac Equation

The Dirac equation describes spin-1/2 fermions in relativistic quantum mechanics:

$$i\hbar \frac{\partial \psi}{\partial t} = \left(-i\hbar c \boldsymbol{\alpha} \cdot \nabla + \beta mc^2\right) \psi$$

In 1+1 dimensions, the two-component spinor satisfies:

$$i\frac{\partial}{\partial t}\begin{pmatrix} \psi_1 \\ \psi_2 \end{pmatrix} = \begin{pmatrix} m & -i\partial_x \\ -i\partial_x & -m \end{pmatrix} \begin{pmatrix} \psi_1 \\ \psi_2 \end{pmatrix}$$

**Applications:** Electrons, quarks, neutrinos (with mass), graphene excitations, relativistic quantum chemistry, magnetic monopoles.

In [ ]:
# Symbolic solution
eq = registry.get('dirac_equation')
print("Dirac Equation:")
print(f"  Form: {eq.description}")
sol = eq.symbolic_solution()
print(f"  General solution: {sol}")

In [ ]:
# Numerical solution: two-component spinor evolution
Nx = 300
Nt = 200
x = np.linspace(-10, 10, Nx)
t = np.linspace(0, 5, Nt)

# Initial Gaussian spinor packet
sigma = 1.0
k0 = 2.0
envelope = np.exp(-x**2 / (2 * sigma**2))
psi1_0 = envelope * np.exp(1j * k0 * x)
psi2_0 = 0.3 * envelope * np.exp(1j * k0 * x)

result = eq.numerical_solve(
    t_span=(0, 5),
    u0=np.column_stack([psi1_0, psi2_0]),
    x=x,
    t_eval=t,
    params={'m': 1.0, 'c': 1.0}
)

# Plot |psi_1| and |psi_2| at several times
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
snapshot_times = [0, len(t)//4, len(t)//2, 3*len(t)//4]

for idx, ti in enumerate(snapshot_times):
    ax = axes[idx // 2][idx % 2]
    psi1 = result.u[ti, :Nx]
    psi2 = result.u[ti, Nx:]
    ax.plot(x, np.abs(psi1), 'b-', linewidth=1.5, label='$|\psi_1|$')
    ax.plot(x, np.abs(psi2), 'r-', linewidth=1.5, label='$|\psi_2|$')
    ax.set_title(f'$t = {t[ti]:.2f}$')
    ax.set_xlabel('$x$')
    ax.set_ylabel('$|\psi|$')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Dirac Equation: Two-Component Spinor Evolution ($m=1$)', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Proca Equation

The Proca equation describes massive spin-1 vector bosons:

$$\partial_\mu F^{\mu\nu} + m^2 A^\nu = 0$$

where $F^{\mu\nu} = \partial^\mu A^\nu - \partial^\nu A^\mu$ and $A^\nu$ is the vector potential. In 1+1 dimensions for the $A^0$ component:

$$\frac{\partial^2 A^0}{\partial t^2} - \frac{\partial^2 A^0}{\partial x^2} + m^2 A^0 = 0$$

**Applications:** W and Z bosons (weak interaction), massive photon in plasma, vector mesons, dark photon candidates.

In [ ]:
# Symbolic solution
eq = registry.get('proca_equation')
print("Proca Equation:")
print(f"  Form: {eq.description}")
sol = eq.symbolic_solution()
print(f"  General solution: {sol}")

In [ ]:
# Numerical solution: massive vector boson propagation
Nx = 300
Nt = 200
x = np.linspace(-15, 15, Nx)
t = np.linspace(0, 8, Nt)

# Initial Gaussian pulse
A0 = np.exp(-x**2 / 3.0)
dA0 = np.zeros(Nx)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, (mass, title) in enumerate([(0.0, '$m = 0$ (Maxwell)'), (1.5, '$m = 1.5$ (Proca)')]):
    result = eq.numerical_solve(
        t_span=(0, 8),
        u0=np.column_stack([A0, dA0]),
        x=x,
        t_eval=t,
        params={'m': mass}
    )
    plot_pde_heatmap(result, ax=axes[idx],
        x_label='$x$', t_label='$t$',
        title=f'Proca Field: {title}')

plt.suptitle('Proca Equation: Massless (Photon) vs Massive (Vector Boson)', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Weyl Equation

The Weyl equation describes massless spin-1/2 particles (chiral fermions):

$$i \frac{\partial \psi}{\partial t} = \pm i \boldsymbol{\sigma} \cdot \nabla \psi$$

In 1+1 dimensions:
$$i\frac{\partial \psi}{\partial t} = i\sigma_1 \frac{\partial \psi}{\partial x}$$

Solutions propagate at the speed of light with definite chirality (handedness).

**Applications:** Neutrinos (massless limit), Weyl semimetals, chiral anomaly, parity violation in weak interactions.

In [ ]:
# Symbolic solution
eq = registry.get('weyl_equation')
print("Weyl Equation:")
print(f"  Form: {eq.description}")
sol = eq.symbolic_solution()
print(f"  General solution: {sol}")

In [ ]:
# Numerical solution: chiral wave propagation
Nx = 300
Nt = 200
x = np.linspace(-10, 10, Nx)
t = np.linspace(0, 4, Nt)

# Initial localized wave
sigma = 0.8
psi_L = np.exp(-x**2 / (2*sigma**2)) * np.exp(1j * 3.0 * x)
psi_R = np.zeros(Nx)

result = eq.numerical_solve(
    t_span=(0, 4),
    u0=np.column_stack([psi_L, psi_R]),
    x=x,
    t_eval=t,
    params={}
)

fig, ax = plt.subplots(figsize=(10, 6))
plot_pde_heatmap(result, ax=ax,
    x_label='$x$', t_label='$t$',
    title='Weyl Equation: Chiral Fermion Propagation')
plt.tight_layout()
plt.show()

## 5. Sine-Gordon Equation

The sine-Gordon equation is a nonlinear PDE:

$$\frac{\partial^2 \phi}{\partial t^2} - \frac{\partial^2 \phi}{\partial x^2} + \sin\phi = 0$$

It supports exact soliton (kink) solutions: $\phi(x,t) = 4\arctan\exp\left(\gamma(x - vt)\right)$ where $\gamma = 1/\sqrt{1-v^2}$.

**Applications:** Josephson junctions, crystal dislocation theory, magnetic domain walls, fiber optics, integrable systems theory.

In [ ]:
# Symbolic solution
eq = registry.get('sine_gordon_equation')
print("Sine-Gordon Equation:")
print(f"  Form: {eq.description}")
sol = eq.symbolic_solution()
print(f"  General solution: {sol}")
print("  Kink soliton: phi(x,t) = 4*arctan(exp(gamma*(x - v*t)))")

In [ ]:
# Numerical solution: kink soliton propagation
Nx = 400
Nt = 200
x = np.linspace(-15, 15, Nx)
t = np.linspace(0, 10, Nt)

# Kink soliton initial condition
v = 0.5  # velocity
gamma = 1.0 / np.sqrt(1.0 - v**2)
phi_kink = 4.0 * np.arctan(np.exp(gamma * x))
# Time derivative of kink
dphi_kink = -4.0 * v * gamma * np.exp(gamma * x) / (1.0 + np.exp(2.0 * gamma * x))

result = eq.numerical_solve(
    t_span=(0, 10),
    u0=np.column_stack([phi_kink, dphi_kink]),
    x=x,
    t_eval=t,
    params={}
)

# Show snapshots
fig, ax = plt.subplots(figsize=(10, 6))
plot_pde_snapshots(result, ax=ax, x_label='$x$', y_label='$\phi(x,t)$',
    title='Sine-Gordon Kink Soliton Propagation', num_snapshots=6)
plt.tight_layout()
plt.show()

## 6. KdV Equation

The Korteweg-de Vries (KdV) equation is:

$$\frac{\partial u}{\partial t} + 6u\frac{\partial u}{\partial x} + \frac{\partial^3 u}{\partial x^3} = 0$$

It admits the one-soliton solution: $u(x,t) = \frac{c}{2} \mathrm{sech}^2\left(\frac{\sqrt{c}}{2}(x - ct)\right)$.

**Applications:** Shallow water waves, ion-acoustic waves in plasma, internal ocean waves, Fermi-Pasta-Ulam recurrence, integrable systems.

In [ ]:
# Symbolic solution
eq = registry.get('kdv_equation')
print("KdV Equation:")
print(f"  Form: {eq.description}")
sol = eq.symbolic_solution()
print(f"  General solution: {sol}")
print("  Soliton: u(x,t) = (c/2) * sech^2(sqrt(c)/2 * (x - c*t))")

In [ ]:
# Numerical solution: KdV soliton
Nx = 400
Nt = 200
x = np.linspace(-20, 30, Nx)
t = np.linspace(0, 5, Nt)

# Soliton initial condition: u(x,0) = (c/2) * sech^2(sqrt(c)/2 * (x - x0))
c = 4.0  # soliton speed (and amplitude parameter)
x0 = -10.0
u0 = (c / 2.0) / np.cosh(np.sqrt(c) / 2.0 * (x - x0))**2

result = eq.numerical_solve(
    t_span=(0, 5),
    u0=u0,
    x=x,
    t_eval=t,
    params={}
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Heatmap
plot_pde_heatmap(result, ax=axes[0],
    x_label='$x$', t_label='$t$',
    title='KdV Soliton: $u(x,t)$')

# Snapshots
plot_pde_snapshots(result, ax=axes[1],
    x_label='$x$', y_label='$u(x,t)$',
    title='KdV Soliton Snapshots', num_snapshots=5)

plt.tight_layout()
plt.show()

## 7. Yang-Mills (SU(2)) Equation

The classical Yang-Mills equations for SU(2) gauge theory in 1+1 dimensions:

$$D_\mu F^{\mu\nu} = 0$$

where $F^{\mu\nu} = \partial^\mu A^\nu - \partial^\nu A^\mu + g[A^\mu, A^\nu]$ and $D_\mu = \partial_\mu + g[A_\mu, \cdot]$.

For spatially homogeneous fields ($A_x = 0$, only $A_0$ non-trivial), the equations reduce to coupled ODEs for three gauge components $A_1, A_2, A_3$ exhibiting chaotic dynamics.

**Applications:** Strong nuclear force (QCD), gauge theory foundations, confinement, asymptotic freedom, lattice gauge theory.

In [ ]:
# Symbolic solution
eq = registry.get('yang_mills_su2')
print("Yang-Mills SU(2) Equations:")
print(f"  Form: {eq.description}")
sol = eq.symbolic_solution()
print(f"  General solution: {sol}")

In [ ]:
# Numerical solution: chaotic self-interaction in 3D phase space
t_span = (0, 50)
t_eval = np.linspace(0, 50, 5000)

# Initial conditions for A1, A2, A3 and their derivatives
y0 = [0.1, 0.5, 0.3, 0.0, 0.1, -0.2]  # [A1, A2, A3, A1', A2', A3']

result = eq.numerical_solve(t_span, y0, t_eval=t_eval,
    params={'g': 1.0})

A1 = result.y[0]
A2 = result.y[1]
A3 = result.y[2]

# 3D phase portrait of A_i components
fig = plt.figure(figsize=(12, 10))

# 3D trajectory
ax3d = fig.add_subplot(2, 2, (1, 2), projection='3d')
ax3d.plot(A1, A2, A3, linewidth=0.3, alpha=0.7, color='#1f77b4')
ax3d.scatter(A1[0], A2[0], A3[0], color='green', s=50, label='Start')
ax3d.set_xlabel('$A_1$')
ax3d.set_ylabel('$A_2$')
ax3d.set_zlabel('$A_3$')
ax3d.set_title('Yang-Mills SU(2): Chaotic Gauge Field Dynamics')
ax3d.legend()

# Time series
ax_ts = fig.add_subplot(2, 2, 3)
ax_ts.plot(result.t, A1, linewidth=0.5, label='$A_1(t)$', color='#1f77b4')
ax_ts.plot(result.t, A2, linewidth=0.5, label='$A_2(t)$', color='#ff7f0e')
ax_ts.plot(result.t, A3, linewidth=0.5, label='$A_3(t)$', color='#2ca02c')
ax_ts.set_xlabel('Time $t$')
ax_ts.set_ylabel('Field components')
ax_ts.set_title('Time Series')
ax_ts.legend()
ax_ts.grid(True, alpha=0.3)

# 2D phase portrait A1 vs A2
ax_phase = fig.add_subplot(2, 2, 4)
ax_phase.plot(A1, A2, linewidth=0.3, alpha=0.7, color='#1f77b4')
ax_phase.set_xlabel('$A_1$')
ax_phase.set_ylabel('$A_2$')
ax_phase.set_title('Phase Portrait: $A_1$ vs $A_2$')
ax_phase.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()